# AURORA — PageIndex Two-Stage Editorial Assistant

**Goal of this demo:** Make the *prompt → retrieved context* pipeline visible end-to-end.

**Architecture:** Two PageIndex trees, both built locally, retrieved in sequence:

1. **Article corpus** (10 EN articles, indexed via PageIndex's local markdown runner)
2. **Writing Guide 2026 V1.1** (PDF, indexed via PageIndex's local PDF runner)

Tree JSONs live in `rag/corpus/`. Build them with `rag/scripts/run_pageindex.py` if missing.

Given a prompt like *"I want to write a piece on X"*, Stage 1 retrieves prior coverage from the article tree, Stage 2 retrieves applicable rules from the Writing Guide tree, and a synthesis call writes a cited editorial brief.


## 1 — Setup

In [ ]:
import os, json, sys
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# add repo root to sys.path so we can import the vendored library as rag.pageindex
REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
assert OPENAI_API_KEY, 'Missing OPENAI_API_KEY in .env'

openai_client = OpenAI(api_key=OPENAI_API_KEY)

CORPUS_DIR = REPO_ROOT / 'rag' / 'corpus'
MODEL = 'gpt-4o'

print('OpenAI client ready. Corpus dir:', CORPUS_DIR)


## 2 — Tree 1: Writing Guide (locally-indexed PDF)

Loads the cached tree from `rag/corpus/writing_guide_tree.json`. To rebuild from the PDF:

```
uv run python rag/scripts/run_pageindex.py --pdf_path "data/Writing Guide 2026-V1.1.pdf"
```


In [ ]:
GUIDE_CACHE = CORPUS_DIR / 'writing_guide_tree.json'
assert GUIDE_CACHE.exists(), f'Missing {GUIDE_CACHE}. Run run_pageindex.py on the Writing Guide PDF first.'

guide_tree = json.loads(GUIDE_CACHE.read_text())
print(f'Loaded guide tree from {GUIDE_CACHE.name} — {len(guide_tree)} top-level sections')


In [ ]:
def _loc(node):
    """Return locator label (page or line), agnostic to tree source."""
    if 'page_index' in node:
        return f"p.{node['page_index']}"
    if 'line_num' in node:
        return f"L{node['line_num']}"
    return '?'

def _summary(node):
    """Return summary text, regardless of which key the source uses."""
    return node.get('summary') or node.get('prefix_summary') or node.get('text', '')

def print_tree(nodes, indent=0):
    for n in nodes:
        prefix = '  ' * indent + ('└─ ' if indent > 0 else '')
        print(f"{prefix}[{n['node_id']}] {n['title']}  ({_loc(n)})")
        if n.get('nodes'):
            print_tree(n['nodes'], indent + 1)

print('🌲 Writing Guide structure:\n')
print_tree(guide_tree)

## 3 — Tree 2: Article corpus (locally-indexed markdown)

Built by `rag/scripts/build_corpus.py` (concat of 10 EN articles) + `rag/scripts/run_pageindex.py --md_path`. Stored in `rag/corpus/corpus_en_structure.json`.


In [ ]:
CORPUS_STRUCT = CORPUS_DIR / 'corpus_en_structure.json'
assert CORPUS_STRUCT.exists(), f'Missing {CORPUS_STRUCT}. Run build_corpus.py + run_pageindex.py first.'

with open(CORPUS_STRUCT) as f:
    corpus_data = json.load(f)

articles_tree = corpus_data['structure']
print(f"doc_name: {corpus_data['doc_name']}")
print(f"line_count: {corpus_data['line_count']}")
print(f'Top-level articles: {len(articles_tree)}')

print('\n🌲 Article corpus structure:\n')
print_tree(articles_tree)


## 4 — Helpers: tree search, retrieval, generation

Two stage-specific instructions tell the LLM how to route inside each tree:

- `CORPUS_INSTRUCTION` — prefer specific sub-sections over article roots; allow cross-article picks.
- `GUIDE_INSTRUCTION` — focus on *how to write*, not the topic; skip meta-sections like Contents.

Summaries are truncated to 300 chars (up from the crash course's 150) — gives the routing LLM more signal when there are many candidate articles.

In [ ]:
SUMMARY_CHARS = 300

def compress_tree(nodes):
    out = []
    for n in nodes:
        entry = {
            'node_id': n['node_id'],
            'title':   n['title'],
            'loc':     _loc(n),
            'summary': _summary(n)[:SUMMARY_CHARS],
        }
        if n.get('nodes'):
            entry['children'] = compress_tree(n['nodes'])
        out.append(entry)
    return out

CORPUS_INSTRUCTION = """\
You are routing inside a corpus of published articles. The author wants to write a NEW piece on the topic in their prompt.
Find prior coverage they should be aware of.

Rules:
- Pick the most SPECIFIC sub-sections (leaf-ish nodes) whose content directly addresses the topic.
- Do NOT pick an article-root node unless the whole article is broadly relevant AND no single sub-section is more specific.
- Pick 2 to 5 nodes total. Bias toward fewer, more on-target picks over a long list.
- It is OK to return picks from multiple articles when the topic spans them."""

GUIDE_INSTRUCTION = """\
You are routing inside the ABN AMRO Writing Guide 2026. The author is about to write a piece on the topic in their prompt.
Find Writing Guide rules that govern HOW to write that piece.

Rules:
- Focus on style, structure, tone, accuracy, and inclusivity rules — not the topic of the prompt.
- Skip meta-sections (root document title, \"Contents\", \"Before we get started\").
- Prefer specific named rules over broad parent sections, but include a parent if no single rule fits.
- Pick 3 to 6 nodes total. Cover style + structure + at least one accuracy/inclusivity rule when applicable."""

def llm_tree_search(query, tree, instruction, model=MODEL):
    """Send query + compressed tree to an LLM. Returns dict with 'thinking' + 'node_list'."""
    prompt = f"""{instruction}

User prompt: {query}

Tree:
{json.dumps(compress_tree(tree), indent=2)}

Reply ONLY in this exact JSON format:
{{
  \"thinking\": \"<step-by-step reasoning>\",
  \"node_list\": [\"node_id1\", \"node_id2\"]
}}"""
    resp = openai_client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
    )
    return json.loads(resp.choices[0].message.content)

def find_nodes_by_ids(tree, target_ids):
    found = []
    for node in tree:
        if node['node_id'] in target_ids:
            found.append(node)
        if node.get('nodes'):
            found.extend(find_nodes_by_ids(node['nodes'], target_ids))
    return found

## 5 — Two-stage retrieval pipeline

Stage 1: route over `articles_tree` → which prior pieces touch the topic?

Stage 2: route over `guide_tree` → which Writing Guide rules apply?

Synthesis: one GPT-4o call that takes both retrieval outputs and writes a cited editorial brief.

In [ ]:
def synthesize(query, article_nodes, guide_nodes, model=MODEL):
    article_ctx = '\n\n---\n\n'.join(
        f"[Article section: '{n['title']}' | {_loc(n)}]\n{n.get('text', _summary(n))[:1500]}"
        for n in article_nodes
    ) or '(no prior coverage retrieved)'
    guide_ctx = '\n\n---\n\n'.join(
        f"[Guide section: '{n['title']}' | {_loc(n)}]\n{n.get('text', _summary(n))[:1500]}"
        for n in guide_nodes
    ) or '(no guide sections retrieved)'

    prompt = f"""You are an editorial assistant for ABN AMRO Insights.
The author wants to write: \"{query}\"

PRIOR COVERAGE (from our article archive):
{article_ctx}

HOUSE STYLE GUIDANCE (from the Writing Guide 2026):
{guide_ctx}

Write a brief in 3 short sections:
(a) **What we've already published** — bullets. Cite as (Article: \"<section title>\").
(b) **Recommended angle** — 2–3 sentences. Pick a fresh angle the existing coverage does not already own.
(c) **Writing Guide rules to follow** — bullets. Cite as (Guide: \"<section title>\", <loc>).

Be concrete. Every claim cites a source."""
    resp = openai_client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return resp.choices[0].message.content

def two_stage_rag(query):
    print('=' * 70)
    print(f'PROMPT: {query}')
    print('=' * 70)

    print('\n┏━━━━ STAGE 1: search article corpus ━━━━')
    s1 = llm_tree_search(query, articles_tree, CORPUS_INSTRUCTION)
    print('🧠 thinking:', s1.get('thinking', '')[:400])
    print('🎯 node_list:', s1.get('node_list', []))
    article_nodes = find_nodes_by_ids(articles_tree, s1.get('node_list', []))
    print(f'📄 retrieved {len(article_nodes)} article sections:')
    for n in article_nodes:
        print(f"   • [{n['node_id']}] {n['title']} ({_loc(n)})")

    print('\n┏━━━━ STAGE 2: search Writing Guide ━━━━')
    s2 = llm_tree_search(query, guide_tree, GUIDE_INSTRUCTION)
    print('🧠 thinking:', s2.get('thinking', '')[:400])
    print('🎯 node_list:', s2.get('node_list', []))
    guide_nodes = find_nodes_by_ids(guide_tree, s2.get('node_list', []))
    print(f'📄 retrieved {len(guide_nodes)} guide sections:')
    for n in guide_nodes:
        print(f"   • [{n['node_id']}] {n['title']} ({_loc(n)})")

    print('\n┏━━━━ SYNTHESIS ━━━━')
    brief = synthesize(query, article_nodes, guide_nodes)
    print(brief)

    return {
        'stage1': s1, 'article_nodes': article_nodes,
        'stage2': s2, 'guide_nodes': guide_nodes,
        'brief': brief,
    }

## 6 — Hero query: agentic-AI piece for SMEs

In [ ]:
hero = two_stage_rag(
    'I want to write a new piece on agentic-AI risks for small and medium businesses. '
    '(1) What have we already published on this? '
    '(2) Which Writing Guide rules apply?'
)

## 7 — Follow-up query: European tech sovereignty

Same pipeline, different topic. Should hit the *Tech Without Google* and *Digital Sovereignty* articles.

In [ ]:
followup = two_stage_rag(
    'We are planning a piece on European digital sovereignty for 2026. '
    'What is our prior coverage, and which Writing Guide rules should the author follow?'
)